In [ ]:
import pathlib
import wave

import numpy as np
import pandas as pd
import xy.pyplot as plt
from scipy.fft import irfft, rfft, rfftfreq
from scipy.io.wavfile import read
from src.note import Note

## Preprocessing

In [ ]:
DATA_DIR = pathlib.Path("./test_files")

In [ ]:
wav_filename = DATA_DIR / "C major.wav"

In [ ]:
def load_wav(
    wav_filename: str | pathlib.Path,
) -> tuple[np.ndarray, int]:
    sampling_freq, samples = read(wav_filename)
    with wave.open(str(wav_filename), "rb") as wav:
        bit_depth = wav.getsampwidth() * 8

    samples: np.ndarray = np.asarray(samples, dtype=float)
    if samples.ndim == 1:
        channel = samples
    else:
        channel = samples.sum(axis=1) / samples.shape[1]

    signal: np.ndarray = channel / 2 ** (bit_depth - 1)
    signal = signal - signal.mean()
    return signal, sampling_freq

In [ ]:
signal, sampling_freq = load_wav(wav_filename)

In [ ]:
def band_pass_filter(
    signal: np.ndarray,
    sampling_freq: int,
    low_freq: float = 20,
    high_freq: float = 5e3,
) -> np.ndarray:
    assert low_freq < high_freq
    frequencies = rfftfreq(signal.size, d=1 / sampling_freq)
    fourier = rfft(signal)
    fourier[frequencies > high_freq] = 0
    fourier[frequencies < low_freq] = 0
    return irfft(fourier, n=signal.size)

In [ ]:
f_signal = band_pass_filter(
    signal,
    sampling_freq,
)

## STFT

In [ ]:
WINDOW_SIZE = 4096
HOP_FRACTION = 1 / 2
HOP_SIZE = int(WINDOW_SIZE * HOP_FRACTION)

In [ ]:
padded_sig = np.pad(
    signal,
    (0, WINDOW_SIZE - len(signal) % WINDOW_SIZE),
)

In [ ]:
def make_segments(
    signal: np.ndarray, window_size=WINDOW_SIZE, hop_size=HOP_SIZE
) -> tuple[list[np.ndarray], list[tuple[int, int]]]:
    starts = np.arange(0, len(signal), hop_size)
    required_length = starts[-1] + window_size
    padded_signal = np.pad(signal, (0, required_length - len(signal)))

    segments = [padded_signal[start : start + window_size] for start in starts]
    indices = [(start, start + window_size) for start in starts]

    return segments, indices

In [ ]:
segments, indices = make_segments(f_signal)

In [ ]:
stft_df = pd.DataFrame(
    {
        "segment": segments,
        "indices": indices,
    },
)
stft_df["timestamps"] = stft_df["indices"].apply(
    lambda x: (x[0] / sampling_freq, x[1] / sampling_freq)
)

In [ ]:
def apply_hann(x: np.ndarray) -> np.ndarray:
    window = np.hanning(len(x))
    return x * window


stft_df["hann_segment"] = stft_df["segment"].apply(apply_hann)

In [ ]:
def fourier_segment(x: np.ndarray) -> np.ndarray:
    f_seg = rfft(x)
    amplitudes = np.abs(f_seg) / len(x) * 4
    return amplitudes

In [ ]:
stft_df["amplitude"] = stft_df["hann_segment"].apply(fourier_segment)

In [ ]:
frequencies = rfftfreq(WINDOW_SIZE, 1 / sampling_freq)
frequencies

## Plotting Spectrogram

In [ ]:
spectrogram = np.array(stft_df["amplitude"].to_list())
spectrogram.shape

In [ ]:
# frame_starts = np.array([index[0] for index in stft_df["indices"]])

# ixes = np.arange(spectrogram.shape[0])
# frequency_limit = 4e3
# frequency_mask = frequencies <= frequency_limit

# spectrogram_for_plot = spectrogram[:, frequency_mask]
# spectrogram_db = 20 * np.log10(
#     np.maximum(spectrogram_for_plot, np.finfo(float).eps)
# )
# spectrogram_db -= spectrogram_db.max()

# fig, ax = plt.subplots(figsize=(20, 10))
# image = ax.pcolormesh(
#     ixes,
#     frequencies[frequency_mask],
#     spectrogram_db.T,
#     shading="auto",
#     cmap="magma",
#     vmin=-80,
# )

# ax.set_title("STFT Spectrogram")
# ax.set_xlabel("Indices")
# ax.set_ylabel("Frequency (Hz)")
# ax.set_ylim(0, frequency_limit)
# fig.colorbar(image, ax=ax, label="Magnitude (dB)")
# plt.show()

In [ ]:
# frame_times = np.array([np.mean(index) / sampling_freq for index in indices])
# frequency_limit = 4e3
# frequency_mask = frequencies <= frequency_limit

# spectrogram_for_plot = spectrogram[:, frequency_mask]
# spectrogram_db = 20 * np.log10(
#     np.maximum(spectrogram_for_plot, np.finfo(float).eps)
# )
# spectrogram_db -= spectrogram_db.max()

# fig, ax = plt.subplots(figsize=(20, 10))
# image = ax.pcolormesh(
#     frame_times,
#     frequencies[frequency_mask],
#     spectrogram_db.T,
#     shading="auto",
#     cmap="magma",
#     vmin = -80,
# )
# ax.set_title("STFT Spectrogram")
# ax.set_xlabel("Time (s)")
# ax.set_ylabel("Frequency (Hz)")
# ax.set_ylim(0, frequency_limit)
# fig.colorbar(image, ax=ax, label="Magnitude (dB)", )
# plt.show()

### Note Detection

## Note Detection

A useful first noise-floor estimate is the local spectral baseline in each STFT frame. For a given frequency bin, take the median magnitude of its neighboring bins. A narrow tonal peak rises above this baseline, while broadband noise tends to be absorbed into it.

The detector below works in decibels and uses `prominence`: the height of a peak relative to the surrounding baseline. The two important parameters are:

- `baseline_width_hz`: make this wider than a single spectral peak, but not so wide that it follows the whole spectrum.
- `prominence_db`: the minimum amount a peak must rise above its local baseline.

This is a local background estimate, not a perfect measurement of physical noise. It is a good first detector because it adapts to frequency-dependent noise and changing volume.

In [ ]:
db_spectrogram = 20 * np.log10(np.clip(spectrogram, a_min=1e-6, a_max=None))

In [ ]:
from scipy.signal import find_peaks

In [ ]:
nothing = spectrogram[24]
g2 = spectrogram[281]

In [ ]:
thing = g2

In [ ]:
peaks, stats = find_peaks(
    x=thing,
    prominence=0.001,
)

In [ ]:
spec = spectrogram[284]
peaks, stats = find_peaks(
    spec,
    height=10e-4,
)

In [ ]:
fs = frequencies[peaks]
amps = spec[peaks]
fs, amps

In [ ]:
def detect_fundamentals(
    frequencies: list[float],
    amplitudes: list[float],
    tolerance_cents: float = 50,
    min_harmonics: int = 2,
    max_harmonic: int = 5,
) -> tuple[list[float], list[dict]]:
    fundamentals = []
    stats = []
    remaining = np.ones(len(frequencies), dtype=bool)

    for candidate_index in range(len(frequencies)):
        if not remaining[candidate_index]:
            continue

        seed_frequency = frequencies[candidate_index]
        candidate_frequency = seed_frequency
        available_indices = np.flatnonzero(remaining)
        matched_indices = []
        matched_harmonic_numbers = set()

        for peak_index in available_indices:
            ratio = frequencies[peak_index] / candidate_frequency
            harmonic_number = round(ratio)

            if not 2 <= harmonic_number <= max_harmonic:
                continue

            cents_error = 1200 * abs(np.log2(ratio / harmonic_number))
            if cents_error > tolerance_cents:
                continue

            matched_indices.append(peak_index)
            matched_harmonic_numbers.add(harmonic_number)
            candidate_frequency = frequencies[peak_index] / harmonic_number

        if len(matched_harmonic_numbers) >= min_harmonics:
            fundamental_amplitude = amplitudes[candidate_index]
            loudness_db = 20 * np.log10(max(fundamental_amplitude, np.finfo(float).eps))
            harmonic_score = len(matched_harmonic_numbers)
            matched_indices = np.asarray(matched_indices, dtype=int)

            fundamentals.append(candidate_frequency)
            stats.append(
                {
                    "frequency": candidate_frequency,
                    "harmonics": sorted(matched_harmonic_numbers),
                    "harmonic_score": harmonic_score,
                    "fundamental_amplitude": fundamental_amplitude,
                    "loudness_db": loudness_db,
                }
            )
        # Remove the candidate and all of its observed harmonic peaks.
        remaining[candidate_index] = False
        remaining[matched_indices] = False
    return fundamentals, stats

In [ ]:
fundamentals, stats = detect_fundamentals(fs, amps)

In [ ]:
C0 = Note(NoteNumber=0, Octave=0)
CANDIDATE_NOTES = [(C0 + i) for i in range(12 * 6)]
CANDIDATE_FREQUENCIES = np.array([note.frequency for note in CANDIDATE_NOTES])


def frequencies_to_notes(frequencies: list[float]) -> list[tuple[Note, float]]:
    ans = []
    for freq in frequencies:
        cents_diff = np.abs(1200 * np.log2(CANDIDATE_FREQUENCIES / freq))
        minidx = np.argmin(cents_diff)
        mindiff = np.min(cents_diff)
        ans.append((CANDIDATE_NOTES[minidx], mindiff))
    return ans

In [ ]:
mynotes = frequencies_to_notes(fundamentals)
[n[0].name for n in mynotes]

In [ ]:
peak_rows = []
note_rows = []

for frame_index, spectrum in enumerate(spectrogram):
    peaks, peak_stats = find_peaks(
        spectrum,
        height=10e-4,
    )

    peak_rows.extend(
        {
            "frame_index": frame_index,
            "time": frame_times[frame_index],
            "frequency": frequencies[peak_index],
            "amplitude": spectrum[peak_index],
        }
        for peak_index in peaks
    )

    frame_frequencies = frequencies[peaks]
    frame_amplitudes = spectrum[peaks]
    frame_fundamentals, frame_stats = detect_fundamentals(
        frame_frequencies,
        frame_amplitudes,
    )
    frame_notes = frequencies_to_notes(frame_fundamentals)

    for fundamental, fundamental_stats, (note, cents_error) in zip(
        frame_fundamentals,
        frame_stats,
        frame_notes,
    ):
        note_rows.append(
            {
                "frame_index": frame_index,
                "time": frame_times[frame_index],
                "frequency": fundamental,
                "note": note.name,
                "cents_error": cents_error,
                "harmonics": fundamental_stats["harmonics"],
                "harmonic_score": fundamental_stats["harmonic_score"],
                "fundamental_amplitude": fundamental_stats["fundamental_amplitude"],
                "loudness_db": fundamental_stats["loudness_db"],
            }
        )

all_peaks = pd.DataFrame(peak_rows)
detected_notes = pd.DataFrame(note_rows)
frequency_mask = frequencies <= frequency_limit
spectrogram_for_plot = spectrogram[:, frequency_mask]
spectrogram_db = 20 * np.log10(np.maximum(spectrogram_for_plot, np.finfo(float).eps))
spectrogram_db -= spectrogram_db.max()

fig, ax = plt.subplots(figsize=(20, 10))
image = ax.pcolormesh(
    frame_times,
    frequencies[frequency_mask],
    spectrogram_db.T,
    shading="auto",
    cmap="magma",
    vmin=-80,
)

visible_peaks = all_peaks[all_peaks["frequency"] <= frequency_limit]
ax.scatter(
    visible_peaks["time"],
    visible_peaks["frequency"],
    s=12,
    color="cyan",
    linewidths=0.8,
    label="Spectral peaks",
)

visible_notes = detected_notes[detected_notes["frequency"] <= frequency_limit]
ax.scatter(
    visible_notes["time"],
    visible_notes["frequency"],
    s=45,
    color="lime",
    marker="x",
    linewidths=1.2,
    label="Detected fundamentals",
)

ax.set_title("Spectral Peaks and Detected Notes")
ax.set_xlabel("Time (s)")
ax.set_ylabel("Frequency (Hz)")
ax.set_ylim(0, frequency_limit)
ax.legend()
fig.colorbar(image, ax=ax, label="Magnitude (dB)")
plt.show()